---

The cell below import all required dependencies; run it if your device lacks any of them.

---

In [2]:
import importlib.util

REQUIRED_PACKAGES = {
    "kaggle": "kaggle",
    "roboflow": "roboflow",
    "yaml": "PyYAML",
    "dotenv": "python-dotenv",
    "PIL": "Pillow",
    "cv2": "opencv-python-headless",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "cvat_sdk": "cvat-sdk"
}

missing_packages = []

for import_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"✗ Missing: {package_name}")
        missing_packages.append(package_name)
    else:
        print(f"✓ Found: {package_name}")

if missing_packages:
    print("\nMissing packages detected:")
    print(f"{' '.join(missing_packages)}")
    print("\nInstall using:")
    print(f"python -m pip install --user {' '.join(missing_packages)}")
else:
    print("\n✅ All dependencies are installed.")

✓ Found: kaggle
✓ Found: roboflow
✓ Found: PyYAML
✓ Found: python-dotenv
✓ Found: Pillow
✓ Found: opencv-python-headless
✓ Found: tqdm
✓ Found: pandas
✓ Found: numpy
✓ Found: matplotlib
✗ Missing: cvat-sdk

Missing packages detected:
cvat-sdk

Install using:
python -m pip install --user cvat-sdk


In [52]:
# Standard libraries
import os
from pathlib import Path
from urllib.parse import urlparse

# Third-party libraries
import yaml

# Dataset APIs
from roboflow import Roboflow
from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

load_dotenv()

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


---

The Kaggle API message can be ignored for now; it will be resolved once we write the Kaggle download function.

---

---

Verify our current location in the project

---

In [53]:
CURRENT_DIR = Path.cwd()

print("Current directory:")
print(CURRENT_DIR)

print("\nContents:")
for item in CURRENT_DIR.iterdir():
    print("-", item.name)

Current directory:
D:\SIT374\WalkBuddy-T2-2026\ML_side\notebooks\data_pipeline

Contents:
- .ipynb_checkpoints
- data_collection.ipynb
- validation.ipynb


---

Starting from this file, we'll backtrack to its parent folder; `parents[0]` is the parent before it, which is "notebooks", but we need to travel back to ML_side, so I used `parents[1]`.

---

In [54]:
ML_SIDE_ROOT = Path.cwd().parents[1]

print("Project root:")
print(ML_SIDE_ROOT)

print("\nContents:")
for item in ML_SIDE_ROOT.iterdir():
    print("-", item.name)

Project root:
D:\SIT374\WalkBuddy-T2-2026\ML_side

Contents:
- config
- datasets
- dataset_analyze.py
- data_pipeline
- deployment
- docs
- experiments
- main.py
- models
- notebooks
- priority_demo_cli.py
- README.md
- src
- testing_pipeline
- tests


---

This block defines the dataset directory paths used throughout the pipeline and displays their locations for verification.

---

In [55]:
DATASET_DIR = ML_SIDE_ROOT / "datasets"

RAW_DIR = DATASET_DIR / "raw"
INVALID_DIR = DATASET_DIR / "invalid"
PROCESSED_DIR = DATASET_DIR / "processed"

print("Dataset paths:")
print("RAW:", RAW_DIR)
print("INVALID:", INVALID_DIR)
print("PROCESSED:", PROCESSED_DIR)

Dataset paths:
RAW: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw
INVALID: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\invalid
PROCESSED: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\processed


---

This block checks whether the required dataset directories exist within the project structure. If a directory is missing, it is created automatically.

---

In [56]:
dataset_folders = [
    RAW_DIR,
    INVALID_DIR,
    PROCESSED_DIR
]

for folder in dataset_folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Created/checked: {folder}")

Created/checked: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw
Created/checked: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\invalid
Created/checked: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\processed


---

This block checks whether the download_sheet.yaml configuration file exists at the expected project location before loading it.

---

In [57]:
CONFIG_DIR = ML_SIDE_ROOT / "config"
CONFIG_FILE = CONFIG_DIR / "download_sheet.yaml"

print("Config file location:")
print(CONFIG_FILE)

print("\nExists:")
print(CONFIG_FILE.exists())

Config file location:
D:\SIT374\WalkBuddy-T2-2026\ML_side\config\download_sheet.yaml

Exists:
True


---

This block loads the YAML configuration file and converts it into a Python dictionary for accessing dataset metadata.

---

In [58]:
with open(CONFIG_FILE, "r") as file:
    dataset_config = yaml.safe_load(file)

print(type(dataset_config))
print(dataset_config.keys())

<class 'dict'>
dict_keys(['datasets'])


---

This block is a check for the configuration file.

---

In [59]:
for dataset_name, info in dataset_config["datasets"].items():
    print(f"\nDataset: {dataset_name}")
    print("Fields:", info.keys())


Dataset: obstacle_detection_roboflow
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: obstacle_detection_kaggle
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: obstacle_detection_huggingface
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: indoor_object_detection
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: mapillary_vistas
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: light_poles
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: fpv_crosswalk_segmentation
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: pedestrian_detection
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

Dataset: road_sign_detection
Fields: dict_keys(['source', 'url', 'version', 'status', 'folder', 'ethics'])

D

---

This block validates that all dataset sources contain the required configuration fields before processing.

---

In [60]:
required_fields = [
    "source",
    "url",
    "version",
    "status",
    "folder",
    "ethics"
]

for dataset_name, info in dataset_config["datasets"].items():
    missing = [
        field for field in required_fields
        if field not in info
    ]

    if missing:
        print(f"❌ {dataset_name} missing: {missing}")
    else:
        print(f"✅ {dataset_name} configuration complete")

✅ obstacle_detection_roboflow configuration complete
✅ obstacle_detection_kaggle configuration complete
✅ obstacle_detection_huggingface configuration complete
✅ indoor_object_detection configuration complete
✅ mapillary_vistas configuration complete
✅ light_poles configuration complete
✅ fpv_crosswalk_segmentation configuration complete
✅ pedestrian_detection configuration complete
✅ road_sign_detection configuration complete
✅ antic_chairs configuration complete
✅ doors_detection configuration complete
✅ indoor_objects_roboflow configuration complete
✅ indoor_detection_vineeth configuration complete
✅ indoor_objects_5iwhq configuration complete
✅ revised_pedestrian_obstacle_detection configuration complete
✅ pedestrian_walk configuration complete
✅ outdoor_objects configuration complete
✅ footpath_detection configuration complete
✅ stairs_detection configuration complete


---

This block categorizes datasets by their source.

---

In [61]:
source_groups = {}

for dataset_name, info in dataset_config["datasets"].items():
    source = info["source"]

    if source not in source_groups:
        source_groups[source] = []

    source_groups[source].append(dataset_name)

for source, datasets in source_groups.items():
    print(f"\n{source}:")
    for dataset in datasets:
        print("-", dataset)


roboflow:
- obstacle_detection_roboflow
- indoor_objects_roboflow
- indoor_detection_vineeth
- indoor_objects_5iwhq
- revised_pedestrian_obstacle_detection
- pedestrian_walk
- outdoor_objects
- footpath_detection
- stairs_detection

kaggle:
- obstacle_detection_kaggle
- indoor_object_detection
- light_poles
- fpv_crosswalk_segmentation
- pedestrian_detection
- road_sign_detection
- antic_chairs
- doors_detection

huggingface:
- obstacle_detection_huggingface

manual:
- mapillary_vistas


---

This block creates the source-based folder structure for each data processing stage. Existing directories are preserved and skipped.

---

In [62]:
DATA_STAGE_DIRS = {
    "raw": RAW_DIR,
    "validated": DATASET_DIR / "validated",
    "interim": DATASET_DIR / "interim"
}

for stage, stage_dir in DATA_STAGE_DIRS.items():
    for source in source_groups.keys():
        source_dir = stage_dir / source
        
        if source_dir.exists():
            print(f"✓ Exists: {source_dir}")
        else:
            source_dir.mkdir(parents=True, exist_ok=True)
            print(f"✅ Created: {source_dir}")

✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\roboflow
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\huggingface
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\manual
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\validated\roboflow
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\validated\kaggle
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\validated\huggingface
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\validated\manual
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\roboflow
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\kaggle
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\huggingface
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\interim\manual


---

This block checks dataset status before processing and handles approved, pending, manual, and unknown statuses safely.

---

In [63]:
valid_statuses = [
    "approved",
    "pending",
    "manual"
]

for dataset_name, info in dataset_config["datasets"].items():
    status = info["status"]

    if status == "approved":
        print(f"✅ {dataset_name}: Approved - ready for processing")

    elif status == "pending":
        print(f"⏸️ {dataset_name}: Pending review - skipped")

    elif status == "manual":
        print(f"📁 {dataset_name}: Manual download required - skipped")

    else:
        print(f"❌ {dataset_name}: Unknown status '{status}' - check YAML")

✅ obstacle_detection_roboflow: Approved - ready for processing
✅ obstacle_detection_kaggle: Approved - ready for processing
❌ obstacle_detection_huggingface: Unknown status 'rejected' - check YAML
✅ indoor_object_detection: Approved - ready for processing
✅ mapillary_vistas: Approved - ready for processing
✅ light_poles: Approved - ready for processing
✅ fpv_crosswalk_segmentation: Approved - ready for processing
✅ pedestrian_detection: Approved - ready for processing
✅ road_sign_detection: Approved - ready for processing
✅ antic_chairs: Approved - ready for processing
✅ doors_detection: Approved - ready for processing
✅ indoor_objects_roboflow: Approved - ready for processing
✅ indoor_detection_vineeth: Approved - ready for processing
✅ indoor_objects_5iwhq: Approved - ready for processing
✅ revised_pedestrian_obstacle_detection: Approved - ready for processing
✅ pedestrian_walk: Approved - ready for processing
✅ outdoor_objects: Approved - ready for processing
✅ footpath_detection: A

---

This block generates the storage path for each dataset based on its source and folder name defined in the YAML configuration.

---

In [64]:
dataset_paths = {}

for dataset_name, info in dataset_config["datasets"].items():

    if info["status"] != "approved":
        print(f"⏭️ Skipping {dataset_name}: {info['status']}")
        continue

    source = info["source"]
    folder = info["folder"]

    dataset_path = RAW_DIR / source / folder
    dataset_paths[dataset_name] = dataset_path

    print(f"✅ {dataset_name}")
    print(f"Path: {dataset_path}")
    print()

✅ obstacle_detection_roboflow
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\roboflow\obstacle_detection_roboflow

✅ obstacle_detection_kaggle
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\obstacle_detection_kaggle

⏭️ Skipping obstacle_detection_huggingface: rejected
✅ indoor_object_detection
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\indoor_object_detection

✅ mapillary_vistas
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\manual\mapillary_vistas

✅ light_poles
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\light_poles

✅ fpv_crosswalk_segmentation
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\fpv_crosswalk_segmentation

✅ pedestrian_detection
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\pedestrian_detection

✅ road_sign_detection
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\road_sign_detection

✅ antic_chairs
Path: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggl

---

This block creates folders for approved datasets.

---

In [65]:
for dataset_name, dataset_path in dataset_paths.items():
    if dataset_path.exists():
        print(f"✓ Exists: {dataset_path}")
    else:
        dataset_path.mkdir(parents=True, exist_ok=True)
        print(f"✅ Created: {dataset_path}")

✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\roboflow\obstacle_detection_roboflow
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\obstacle_detection_kaggle
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\indoor_object_detection
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\manual\mapillary_vistas
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\light_poles
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\fpv_crosswalk_segmentation
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\pedestrian_detection
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\road_sign_detection
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\antic_chairs
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\kaggle\doors_detection
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_side\datasets\raw\roboflow\indoor_objects_roboflow
✓ Exists: D:\SIT374\WalkBuddy-T2-2026\ML_s

---

This block verifies whether all API keys exist in the .env configuration file.

---

In [66]:
keys = {
    "KAGGLE_API_TOKEN": os.getenv("KAGGLE_API_TOKEN"),
    "ROBOFLOW_API_KEY": os.getenv("ROBOFLOW_API_KEY"),
    "HF_TOKEN": os.getenv("HF_TOKEN")
}

for name, value in keys.items():
    if value:
        print(f"✅ {name} detected")
    else:
        print(f"❌ {name} missing")

✅ KAGGLE_API_TOKEN detected
✅ ROBOFLOW_API_KEY detected
✅ HF_TOKEN detected


---

This block loads stored API credentials and creates authenticated clients for external dataset sources.

---

In [67]:
# Load API credentials
KAGGLE_TOKEN = os.getenv("KAGGLE_API_TOKEN")
ROBOFLOW_KEY = os.getenv("ROBOFLOW_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

# Initialize API clients
kaggle_api = KaggleApi()
kaggle_api.authenticate()

roboflow_client = Roboflow(
    api_key=ROBOFLOW_KEY
)

print("✅ All API clients initialized")

✅ All API clients initialized


---

The download process starts with Kaggle, where I create a function that downloads dataset from Kaggle using the API key.

---

In [68]:
def download_kaggle_dataset(dataset_id, destination):

    # Check if dataset already exists
    if destination.exists() and any(destination.iterdir()):
        print(f"⏭️ Skipping {dataset_id}")
        print("Reason: dataset already exists")
        return "skipped"


    print(f"⬇️ Downloading {dataset_id}")

    try:
        kaggle_api.dataset_download_files(
            dataset_id,
            path=destination,
            unzip=True
        )

        # Verify download
        files = list(destination.rglob("*"))

        if len(files) == 0:
            print("❌ Download failed: folder empty")
            return "failed"

        print(f"✅ Download completed: {len(files)} files")
        return "success"


    except Exception as e:
        print(f"❌ Kaggle download error: {e}")
        return "failed"

---

This block finds all approved Kaggle datasets from the YAML configuration and downloads them into their assigned raw folders. Existing datasets are skipped.

---

In [69]:
for dataset_name, info in dataset_config["datasets"].items():

    # Skip non-approved datasets
    if info["status"] != "approved":
        print(f"⏭️ Skipping {dataset_name}: status is {info['status']}")
        continue

    # Only process Kaggle datasets
    if info["source"] != "kaggle":
        continue

    kaggle_id = info["url"].replace(
        "https://www.kaggle.com/datasets/",
        ""
    )

    print("\n" + "=" * 50)
    print(f"Dataset: {dataset_name}")

    result = download_kaggle_dataset(
        kaggle_id,
        dataset_paths[dataset_name]
    )

    print(f"Result: {result}")


Dataset: obstacle_detection_kaggle
⏭️ Skipping abtinzandi/obstacle-detection-dataset
Reason: dataset already exists
Result: skipped
⏭️ Skipping obstacle_detection_huggingface: status is rejected

Dataset: indoor_object_detection
⏭️ Skipping thepbordin/indoor-object-detection
Reason: dataset already exists
Result: skipped

Dataset: light_poles
⏭️ Skipping samuelayman/light-poles
Reason: dataset already exists
Result: skipped

Dataset: fpv_crosswalk_segmentation
⏭️ Skipping zubayearkhaled/first-person-view-crosswalk-segmentation-dataset
Reason: dataset already exists
Result: skipped

Dataset: pedestrian_detection
⏭️ Skipping smeschke/pedestrian-dataset
Reason: dataset already exists
Result: skipped

Dataset: road_sign_detection
⏭️ Skipping andrewmvd/road-sign-detection
Reason: dataset already exists
Result: skipped

Dataset: antic_chairs
⏭️ Skipping arminajdehnia/antic-chairs
Reason: dataset already exists
Result: skipped

Dataset: doors_detection
⏭️ Skipping dataclusterlabs/doors-doors

---

Run this block and double check with the YAML configuration file for Roboflow sources before you proceed with any download.

---

In [73]:
roboflow_versions = {}

for dataset_name, info in dataset_config["datasets"].items():

    if info["status"] != "approved":
        continue

    if info["source"] != "roboflow":
        continue

    url = info["url"]

    # Extract workspace and project from URL
    parts = urlparse(url).path.strip("/").split("/")

    workspace_name = parts[0]
    project_name = parts[1]

    print("\n" + "=" * 50)
    print(f"Dataset: {dataset_name}")
    print(f"Workspace: {workspace_name}")
    print(f"Project: {project_name}")

    try:
        project = roboflow_client.workspace(
            workspace_name
        ).project(
            project_name
        )

        versions = project.versions()

        # Get latest version number
        latest_version = max(
            int(v.version)
            for v in versions
        )

        roboflow_versions[dataset_name] = latest_version

        print(f"✅ Latest version: {latest_version}")

    except Exception as e:
        print(f"❌ Failed: {e}")


Dataset: obstacle_detection_roboflow
Workspace: visually-impaired-obstacle-detection-uxdze
Project: obstacle-detection-yeuzf
loading Roboflow workspace...
loading Roboflow project...
✅ Latest version: 11

Dataset: indoor_objects_roboflow
Workspace: roman-sutter
Project: indoor-objects
loading Roboflow workspace...
loading Roboflow project...
✅ Latest version: 2

Dataset: indoor_detection_vineeth
Workspace: vineeth-optimus
Project: indoor-9xgfb
loading Roboflow workspace...
loading Roboflow project...
✅ Latest version: 1

Dataset: indoor_objects_5iwhq
Workspace: indoor-objects
Project: indoor-5iwhq
loading Roboflow workspace...
loading Roboflow project...
✅ Latest version: 2

Dataset: revised_pedestrian_obstacle_detection
Workspace: thesis-test-dataset
Project: revised-pedestrian-obstacle
loading Roboflow workspace...
loading Roboflow project...
✅ Latest version: 4

Dataset: pedestrian_walk
Workspace: objection-detection
Project: pedestrian-walk
loading Roboflow workspace...
loading Ro

---

This block create the download function for all Roboflow sources.

---

In [71]:
def download_roboflow_dataset(workspace, project_name, version, fmt, destination, dataset_name):
    print("\n" + "=" * 50)
    print(f"Dataset: {dataset_name}")
    print(f"Workspace: {workspace}")
    print(f"Project: {project_name}")
    print(f"Version: {version}")

    # Skip if already downloaded
    if destination.exists() and any(destination.iterdir()):
        print(f"⏭️ Skipping {dataset_name}")
        print("Reason: dataset already exists")
        return "skipped"

    destination.mkdir(parents=True, exist_ok=True)

    zip_path = destination / "roboflow.zip"
    if zip_path.exists():
        print("Removing old Roboflow ZIP")
        zip_path.unlink()

    try:
        print("⬇️ Starting Roboflow download...")
        roboflow_client.workspace(workspace).project(project_name).version(version).download(
            fmt, location=str(destination), overwrite=True
        )

        print("✅ Roboflow download finished")
        files = list(destination.rglob("*"))

        if len(files) == 0:
            print("❌ Download failed: folder empty")
            return "failed"

        print(f"Files found: {len(files)}")
        return "success"

    except Exception as e:
        print(f"❌ Roboflow error: {e}")
        return "failed"

---

This block loop through all approved Roboflow sources and download them, along with skipping folders that already has data in them.

---

In [72]:
for dataset_name, info in dataset_config["datasets"].items():
    if info["status"] != "approved" or info["source"] != "roboflow":
        continue

    workspace, project_name = urlparse(info["url"]).path.strip("/").split("/")

    result = download_roboflow_dataset(
        workspace, project_name, info["version"], info.get("format", "yolov8"),
        dataset_paths[dataset_name], dataset_name
    )
    print(f"Result: {result}")


Dataset: obstacle_detection_roboflow
Workspace: visually-impaired-obstacle-detection-uxdze
Project: obstacle-detection-yeuzf
Version: 11
⏭️ Skipping obstacle_detection_roboflow
Reason: dataset already exists
Result: skipped

Dataset: indoor_objects_roboflow
Workspace: roman-sutter
Project: indoor-objects
Version: 2
⏭️ Skipping indoor_objects_roboflow
Reason: dataset already exists
Result: skipped

Dataset: indoor_detection_vineeth
Workspace: vineeth-optimus
Project: indoor-9xgfb
Version: 1
⏭️ Skipping indoor_detection_vineeth
Reason: dataset already exists
Result: skipped

Dataset: indoor_objects_5iwhq
Workspace: indoor-objects
Project: indoor-5iwhq
Version: 2
⏭️ Skipping indoor_objects_5iwhq
Reason: dataset already exists
Result: skipped

Dataset: revised_pedestrian_obstacle_detection
Workspace: thesis-test-dataset
Project: revised-pedestrian-obstacle
Version: 4
⏭️ Skipping revised_pedestrian_obstacle_detection
Reason: dataset already exists
Result: skipped

Dataset: pedestrian_walk


---
Before we conclude the collection process, this is a work of a junior member, who tried his best to complete a fully automated collection process. If the person working on this found any errors or improvements, please message s224874599@deakin.edu.au

<br>

---

<center>
    End of collection notebook
</center>

---

<br>